### Purpose
Get the initial 1-D background subtraction for just 1 longitude slide-- do a sliding median filter to get it working in the single dimension first before trying to do it for 2 dimensions like was being done before

### Imports

In [ ]:
import altitude_helper
import skymap_data_helper

import importlib
importlib.reload(altitude_helper)
from altitude_helper import *

# for interpolation
from scipy.interpolate import griddata
from scipy.stats import pearsonr
from PIL import Image
from scipy.stats import pearsonr # correlation

import os # folder stuff 

import threading
import time 

from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from functools import partial
from tqdm.notebook import tqdm # since in jupyter

from matplotlib.path import Path # for polygon bounding box 

import scipy.signal # for sliding 1D median filter
from scipy.stats import norm, skewnorm # try to fit gaussian to slice

### Data Loading

In [ ]:
#load an hour of data
site_yknf = 'yknf'
site_fsmi = 'fsmi'
date = datetime(2024,8,30)
hour = 5 #this is in UT

rgb_asi_skymap_lookup_df = skymap_data_helper.build_rgb_asi_skymap_lookup_table(directory='./trex-rgb-asi_data') #CHANGE TO YOUR SKYMAP DIRECTORY!
yknf_rgb_asi_ds = skymap_data_helper.load_rgb_asi_hour_to_xarray(site_yknf, date, hour, rgb_asi_skymap_lookup_df,data_dir='./trex-rgb-asi_data', skymap_dir='./trex-rgb-asi_data') #CHANGE DIRECTORIES!
fsmi_rgb_asi_ds = skymap_data_helper.load_rgb_asi_hour_to_xarray(site_fsmi, date, hour, rgb_asi_skymap_lookup_df,data_dir='./trex-rgb-asi_data', skymap_dir='./trex-rgb-asi_data') #CHANGE DIRECTORIES!


### Helper Functions

In [ ]:
''' Fix number of projecting points at 80km, and project upwards from there so have constant number of longitudes and latitudes'''
# same as "new_get_lon_intensity_slice", just removed the "new" so no confusion
def get_lon_intensity_slice(lat_proj, lon_proj, rgb, 
                        lat_max_box, lat_min_box, lon_max_box, lon_min_box, # dont actually need these boxes, just placeholder so can get function to work
                         lat_camera, lon_camera,
                         site_name, time_index, 
                         og_h, new_h,
                         global_lon_arr, global_lat_arr):

    """
    Given projected latitude and longitude arrays and a particular longitude to slice at (put this particular longitude is relative to 100km proj,
    put it into global_lon_arr as a single element array)
    Find the intensities for the latitudes along this longitude slice  (projected up to new_h, common across both cameras).
    Baseline global_lon_arr and global_lat_arr from 100km (og_h), then projected upwards to new_h. 

    Need to put baseline at 100km because want to see broad range of how the fsmi and yknf peak 
    Use 247.95 degrees as baseline. 

    input:
        lat_proj = 2D projected latitude arr (output of project_lat_lon())
        lon_proj = 2D projected longitude arr (output of project_lat_lon())
        rgb = 3D rgb array (output of mod_plot_lat_lon()) (these are the raw rgb values from the skymap, need to be matched to the projected new projected lat/lon
        time_index = for naming purposes when plotting 
        site_name = for naming purposes when plotting
        og_h = 150,000 km the baseline for where we determined the best longitude slices and the best lat/lon box
        new_h = new height we've projected to for the interpolation (what lat_proj and lon_proj were projected to) 

    output:
        R_peak_lat_arr = 1D arr of all the latitudes that the R-channel had peak altitude for, interpolated over the different longitudes + restricted to (lat_min_box, lat_max_box)
        G_peak_lat_arr = 1D arr of all the latitudes that the G-channel had peak altitude for, interpolated over the different longitudes + restricted to (lat_min_box, lat_max_box)
        B_peak_lat_arr = 1D arr of all the latitudes that the B-channel had peak altitude for, interpolated over the different longitudes + restricted to (lat_min_box, lat_max_box)
        lon_arr = 1D arr of all the longitudes corresponding in idx to the peak latitude arrays (should be same length as the other 3 returned arrays)
    """

    print(f"\n====={new_h/1000.0}km PROJECTION=======")
    print(f"Total {len(global_lon_arr)} longitudes, and for each of these longitudes have   {len(global_lat_arr)} latitudes to interpolate\n")

    # separate into R channel
    R = rgb[:,:, 0]

    # preallocate the sizes of the R_peak_lat_array and R_lon_arr (don't <continue> when run into issue with longitude, rather just add nan in that place)
    R_peak_lat_arr = np.full(len(global_lon_arr), np.nan)
    R_lon_arr = np.full(len(global_lon_arr), np.nan)

    # reproject the latitude slices and the bounding box for each projection --> already looping through all the lon slices here
    # don't need the bounding box for this purpose 
    reproj_lat_arr_dict, reproj_lon_arr_dict, reproj_left_lat, reproj_left_lon, reproj_right_lat, reproj_right_lon, reproj_bottom_lat, reproj_bottom_lon, reproj_top_lat, reproj_top_lon = project_lon_slices_and_box(global_lat_arr, global_lon_arr, # these are in degrees
                                                                                                                                                                                                                           lat_max_box, lat_min_box, lon_max_box, lon_min_box, # also degrees
                                                                                                                                                                                                                           lat_camera, lon_camera, og_h, new_h)
    # LEAVE THIS WHITESPACE ALONE
    original_lon_arr = reproj_lat_arr_dict.keys() # these keys should be the same as take from reproj_lon_arr_dict, and in degrees
    reproj_lon_slice_arr = reproj_lon_arr_dict.values() # this is array of arrays of longitude corresp. to each longitude of the original slices (should just be 1 value)
    reproj_lat_slice_arr = reproj_lat_arr_dict.values() # this is array of latitude for each longiutde in reproj_lon_slice


    # preallocate
    num_slices = len(original_lon_arr)
    R_peak_lat_arr = np.full(num_slices, np.nan)
    R_lon_arr = np.full(num_slices, np.nan)# longitudes that correspond to the peak latitude 
    lon_buffer = 10 + (new_h / 100000)
    for idx, (original_lon,reproj_lat_slice, reproj_lon_slice) in enumerate(zip(original_lon_arr, reproj_lat_slice_arr, reproj_lon_slice_arr)):
        # restrict reproj_lon/lat_slice to only be within the bounding box (don't care outside of the bounding box
        #print(f"lat len b4 slice: {len(reproj_lat_slice)}")
        #print(f"lon len b4 slice: {len(reproj_lon_slice)}")
        slice_in_box_mask = (reproj_lat_slice >= np.min(reproj_bottom_lat)) & (reproj_lat_slice <= np.min(reproj_top_lat))
        reproj_lat_slice = reproj_lat_slice[slice_in_box_mask]
        reproj_lon_slice = reproj_lon_slice[slice_in_box_mask]

        
        # plot the reprojected longitude slice line and the reprojected bounding box
        alpha = 5
        beta = 5
        rgb_adjusted = cv2.convertScaleAbs(rgb, alpha=alpha, beta=beta) # make aurora easier to see on background
        
        altitude_helper.plot_lon_slice_bounding_box(lat_proj, lon_proj, 
                                                    reproj_lat_slice, reproj_lon_slice, #slice
                                                    reproj_left_lat, reproj_left_lon, reproj_right_lat, reproj_right_lon, #bounding box
                                                    reproj_bottom_lat, reproj_bottom_lon, reproj_top_lat, reproj_top_lon, #bounding box
                                                    rgb_adjusted, time_index, site_name, new_h, 
                                                    picket=False) # set picket=False so no zoomed in plotting view
        
        slice_mask = np.abs(lon_proj - original_lon) <= lon_buffer # might need to change this threshold depending on warp in projection
        lon_slice_points = lon_proj[slice_mask]
        lat_slice_points = lat_proj[slice_mask]
        R_slice_values = R[slice_mask] #already flattened 

        # get lat and lon slices over which to interpolate in the right shape 
        flattened_points = np.column_stack((lat_slice_points.flatten(), lon_slice_points.flatten())) # (N,2) pairs of (lat, lon)

        # masks for NaNs in R, G, and B channels separately (basically remove the NaNs and infinite values if there is) --> where the projected lat, lon, or rgb values corresponding to pixel are NaNs
        nan_mask_R = (np.isfinite(flattened_points[:,0]) & np.isfinite(flattened_points[:,1]) & np.isfinite(R_slice_values))
        points_R_clean = flattened_points[nan_mask_R]
        values_R_clean = R_slice_values[nan_mask_R]
        #print(f"Lon {original_lon}: Points captured by mask: {len(points_R_clean)}")

        # get interpolation locations (these are just the (lat,lon) pairs from the reprojected longitude slices
        interp_locations = np.column_stack((reproj_lat_slice, reproj_lon_slice))


        if len(points_R_clean) >= 3 and len(values_R_clean) >= 3:   # minimum for 2D linear interpolation
            R_intensity_profile = griddata(points_R_clean, values_R_clean, interp_locations, method='linear') #intensities along the interp_locations
            # also bound the intensities along the lon slice to only look at those points in the bounding box (as done with reproj_lat/lon_slice above) <-- this is automatically done because intensities are being interpolated onto the defined lat/lon grid which were bounded above!
           # R_intensity_profile = R_intensity_profile[slice_in_box_mask]
        else:
            R_intensity_profile = np.full(len(interp_locations), np.nan)  # or 0

   # print(f"reproj lat slice arr: {reproj_lat_slice_arr}")
    #lat_grid = list(reproj_lat_arr_dict.values())[0]
    lat_grid = reproj_lat_slice
    # print(f"{site_name}{new_h} lat grid: {yknf_lat_grid}")
    # print(f"{site_name}{new_h} intensity grid: {R_intensity_profile}")
    
    return lat_grid, R_intensity_profile
    

In [ ]:
def plot_lon_intensity_slice(site, lat_grid, R_intensity_interp, lon, h):
    """ 
    Given the latitudes and the R channel pixel intensities along some longitude slice for just one site, 
    make a plot showing how the intensity varies across the latitudes for one slice.

    inputs:
        site: string of site name for plot title (YKNF, FSMI)
        lat_grid: array of latitudes we interpolated to in get_intensity_slice for one site (yknf or fsmi)
        R_intensity_interp: array of intensities for each of the latitudes above (len of R intensity = len lat grid)
        lon: the longitude slice for which we stepped along and looked at the brightnesses at diff latitudes
        h: altitude projected to for this longitude slice

    outputs:
        plot
    """
    plt.figure(figsize=(8, 5))
    
    # Plot YKNF intensity profile
    plt.plot(np.array(yknf_lat_grid).flatten(), np.array(yknf_R_intensity_interp).flatten(), label='YKNF', color='red', marker='o', linestyle='-')
        
    plt.xlabel("Latitude (deg)")
    plt.ylabel(f"R-channel intensity")
    plt.title(f"{site} {h/1000.0}km: R-channel intensity vs Latitude at longitude {lon:.2f}°")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_lon_intensity_slices(yknf_lat_grid, yknf_R_intensity_interp,
                        fsmi_lat_grid, fsmi_R_intensity_interp,
                        lon, h):
    """ 
    Given the latitudes and the R channel pixel intensities along some longitude slice for both yknf and fsmi, 
    make a plot showing how the intensity varies across the latitudes for each of the fsmi and yknf slices

    inputs:
        yknf_lat_grid: array of latitudes we interpolated to in get_intensity_slice for yknf
        yknf_R_intensity_interp: array of intensities for each of the latitudes above (len of R intensity = len lat grid)
        fsmi_lat_grid: equiv.
        fsmi_R_intensity_interp: equiv.
        lon: the longitude slice for which we stepped along and looked at the brightnesses at diff latitudes
        h: altitude projected to for this longitude slice

    outputs:
        plot
    """
    plt.figure(figsize=(8, 5))
    
    # Plot YKNF intensity profile
    plt.plot(np.array(yknf_lat_grid).flatten(), np.array(yknf_R_intensity_interp).flatten(), label='YKNF', color='red', marker='o', linestyle='-')
    
    # Plot FSMI intensity profile
    plt.plot(np.array(fsmi_lat_grid).flatten(), np.array(fsmi_R_intensity_interp).flatten(), label='FSMI', color='blue', marker='x', linestyle='--')
    
    plt.xlabel("Latitude (deg)")
    plt.ylabel("R-channel intensity")
    plt.title(f"{h/1000.0}km: R-channel intensity vs Latitude at longitude {lon:.2f}°")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    

### Setup

In [ ]:
time_index = 251
h_target = 200000
site_name_yknf = "Yellowknife"

time_idx_back = 5
R_yknf_back = yknf_rgb_asi_ds.image.sel(channel="R").isel(times=time_idx_back).values
G_yknf_back = yknf_rgb_asi_ds.image.sel(channel="G").isel(times=time_idx_back).values
B_yknf_back = yknf_rgb_asi_ds.image.sel(channel="B").isel(times=time_idx_back).values
rgb_yknf_back = np.stack([R_yknf_back, G_yknf_back, B_yknf_back], axis=-1)


In [ ]:
lat_cam_yknf = yknf_rgb_asi_ds.attrs["site_latitude"]
lon_cam_yknf = yknf_rgb_asi_ds.attrs["site_longitude"]
full_elevation_yknf = yknf_rgb_asi_ds["elevation"]
full_azimuth_yknf = yknf_rgb_asi_ds["azimuth"]


# needed to get spherical intensity slice plots; want a box that covers BOTH of these areas (aka union of yknf and fsmi)
# fsmi
# latmin: 60.5
# latmax: 67.5
# lonmin: 241.5
# lonmax: 262.5

# yknf:
# latmin: 60
# latmax: 68.5
# lonmin: 240
# lonmax: 265

lat_box_min = 60
lat_box_max = 68.5
lon_box_min = 240
lon_box_max = 265


# reference longitude grid to use for the rest of the altitude projections (union of the 2 camera grids)
# want the reference altitude to be somehwere in the middle of all altitudes we are testing for; code will project up and down 
# need to change the initial lat/lon box min/max accordingly 
H_REF = 200000 # this should correspond to the altitude projection used to determine the lat_box_min, etc. above
yknf_lat_ref, yknf_lon_ref = altitude_helper.new_spherical_project_lat_lon(
    full_azimuth_yknf, full_elevation_yknf,
    lat_cam_yknf, lon_cam_yknf,
    H_REF
)

lat_step = 0.5
GLOBAL_LAT_MIN = int(np.floor(np.nanmin(yknf_lat_ref)))


GLOBAL_LAT_MAX = int(np.ceil(np.nanmax(yknf_lat_ref)))


# array of corresp. latitudes to slice at and get the max intensity --> is projected upwards for each longitude slice 
global_lat_arr = np.arange(
    GLOBAL_LAT_MIN,
    GLOBAL_LAT_MAX + lat_step,
    lat_step
)

#--------project chosen az/el to get longitude to slice at for the different altitudes---------#
# working az, el: 85, 40
az_guess_yknf = np.array([90.0]) #deg --> these are eyeballed from yknf plot, so need to project using lat/lon of yknf! could do equiv for fsmi as a sanity check
el_guess_yknf = np.array([70.0]) #deg



# get the fixed geographic anchor point at H_REF
_, lon_ref_anchor = altitude_helper.new_spherical_project_lat_lon(
    az_guess_yknf, el_guess_yknf, lat_cam_yknf, lon_cam_yknf, H_REF
)
lon_anchor = lon_ref_anchor[0]    
print(f"lon anchor: {lon_anchor}")

global_lon_arr = [lon_anchor] # MODIFIED SO NOT ALREADY PROJECTING (shifting in the box was caused by projecting TWICE! once here and once in project_lon_slices_and_box (all projection happening in that func now)

#--------project yknf to h_target-----------#
#----- YKNF ------ #
yknf_lat_proj_arr, yknf_lon_proj_arr = altitude_helper.new_spherical_project_lat_lon(full_azimuth_yknf, 
                                                                     full_elevation_yknf,
                                                                     lat_cam_yknf,
                                                                     lon_cam_yknf,
                                                                     h_target,
                                                                    )

#--------get rgb values for yknf and fsmi for plotting-----------#
time_index = 251
R_yknf = yknf_rgb_asi_ds.image.sel(channel="R").isel(times=time_index).values
G_yknf = yknf_rgb_asi_ds.image.sel(channel="G").isel(times=time_index).values
B_yknf = yknf_rgb_asi_ds.image.sel(channel="B").isel(times=time_index).values
rgb_yknf = np.stack([R_yknf, G_yknf, B_yknf], axis=-1)

yknf_lat_grid, yknf_R_intensity_interp = get_lon_intensity_slice(yknf_lat_proj_arr, yknf_lon_proj_arr, rgb_yknf, 
                                                             lat_box_max, lat_box_min, lon_box_max, lon_box_min,
                                                             lat_cam_yknf, lon_cam_yknf,
                                                             "YKNF", time_index, 
                                                             H_REF, h_target,
                                                             global_lon_arr, global_lat_arr)


lon_to_slice = lon_anchor
#---------- overlaid yknf and fsmi plot of r channel intensity over latitude at a given projected-up longitude------------#
plot_lon_intensity_slice("YKNF", yknf_lat_grid, yknf_R_intensity_interp, lon_to_slice, h_target)



In [ ]:
print(yknf_R_intensity_interp)
print(yknf_lat_grid)

### 1D Median Sliding Filter

In [ ]:
def median_1d_filter_plot(y_values, x_values, site_name, window_size):
    """
        input:
            y_values: typeically R_intensity_interp, the values that you want the median filter to go through
            x_values: typically lat_grid, the latitudes corresponding to the intensity values (y_values) you want the median filter to go through
            site_name: name of site 
            window_size: how many points to look at to calculate the median at once, should be ~2x window of where aurora actually is (this window keeps sliding til you get 1 median corresponding to every point in your y_value array, which you can then subtract out)

        output:
            plot
            median_filter_arr: values outputed from the median filter (to be subtracted from y_values)
    """
    median_filter_arr = scipy.signal.medfilt(y_values, kernel_size=window_size)
    plt.plot(x_values, y_values, marker="o", color="green", label="original intensities")
    plt.plot(x_values, median_filter_arr, marker="o", color="blue", label=f"median intensities (win={window_size})")
    plt.plot(x_values, y_values - median_filter_arr, marker="o", color="red", label="subtracted intensities")
    plt.plot(x_values, np.clip(y_values - median_filter_arr, 0, None), marker="o", color="orange", label="clipped subtracted intensities")
    plt.title(f"{site_name} {h_target/1000} km: slice at lon{lon_to_slice:.1f} (bounded)")
    plt.xlabel("Latitude")
    plt.ylabel("R Intensity")
    plt.legend(
        fontsize="small",
        labelspacing=0.2,
        bbox_to_anchor=(1.05, 1),  
        loc="upper left"           
    )

    plt.show()

    return median_filter_arr

median_1d_filter_plot(yknf_R_intensity_interp, yknf_lat_grid, "YKNF", 7)


In [ ]:
len(yknf_R_intensity_interp)

### Steps
- modify new_get_lon_intensity_slice() for my purposes, plot for sanity check & to see how large window should be
  - see if any changes from newer code needs to be implemented, ie using the proper
- project yknf lat/lon arrays to 200km (as a baseline, and choose a proper longitude to slice at that goes through good clear part of aurora)
- plug in those projected values to get_lon_intensity_slice()
- get the array of intensities per latitude for the one longitude slice
- apply sliding median filter to get new array of medians
  - strategy for window width >= 2 * (duration of intensity spike where aurora is) + 1
  - play around with this to get it working for the single case
- background subtracted intensities = (og intensities) - (array of median intensities) <-- should be the same length
  - plot for sanity check, and keep tweaking method with window width, mean, etc until background subtraction makes sense
- 

brian said: nice to get altitude estimate for particular longitudes as well --> try looping through multiple altitudes at the same longitude and just go off of absolute differences between the max after background subtraction to see what the proper altitude should be for that slice

### Altitude determination for One Longitude Slice
- loop through possible altitudes of steve
- project to each of these altitudes for yknf and fsmi at just one fixed longitude slice, derived from (az,el) = (90, 70)
- perform 1D background subtraction on that slice, take the absolute largest value intensity value, and pull that corresponding latitude as the latitudes to compare between YKNF and FSMI (try an absolute differene and squared difference)
- take the altitude projection that has the lowest difference as the "winner"

In [ ]:
# all possible altitudes of STEVE
h_target_arr = [130000, 140000, 150000, 160000, 170000, 180000, 190000, 200000, 210000, 220000, 230000, 240000, 250000, 260000, 270000, 280000, 290000, 300000]


site_name_yknf = "Yellowknife"
site_name_fsmi = "Fort Smith"


# rgb values for plotting aurora images 
time_index = 251

R_yknf = yknf_rgb_asi_ds.image.sel(channel="R").isel(times=time_index).values
G_yknf = yknf_rgb_asi_ds.image.sel(channel="G").isel(times=time_index).values
B_yknf = yknf_rgb_asi_ds.image.sel(channel="B").isel(times=time_index).values
rgb_yknf = np.stack([R_yknf, G_yknf, B_yknf], axis=-1)

R_fsmi = fsmi_rgb_asi_ds.image.sel(channel="R").isel(times=time_index).values
G_fsmi = fsmi_rgb_asi_ds.image.sel(channel="G").isel(times=time_index).values
B_fsmi = fsmi_rgb_asi_ds.image.sel(channel="B").isel(times=time_index).values
rgb_fsmi = np.stack([R_fsmi, G_fsmi, B_fsmi], axis=-1)

alpha = 5
beta = 5
rgb_yknf_adjusted = cv2.convertScaleAbs(rgb_yknf, alpha=alpha, beta=beta)
rgb_fsmi_adjusted = cv2.convertScaleAbs(rgb_fsmi, alpha=alpha, beta=beta)


# values needed for projection for altitude 
lat_cam_yknf = yknf_rgb_asi_ds.attrs["site_latitude"]
lon_cam_yknf = yknf_rgb_asi_ds.attrs["site_longitude"]
full_elevation_yknf = yknf_rgb_asi_ds["elevation"]
full_azimuth_yknf = yknf_rgb_asi_ds["azimuth"]

lat_cam_fsmi = fsmi_rgb_asi_ds.attrs["site_latitude"]
lon_cam_fsmi = fsmi_rgb_asi_ds.attrs["site_longitude"]
full_elevation_fsmi = fsmi_rgb_asi_ds["elevation"]
full_azimuth_fsmi = fsmi_rgb_asi_ds["azimuth"]


lat_box_min = 60
lat_box_max = 68.5
lon_box_min = 240
lon_box_max = 265

yknf_lat_ref, yknf_lon_ref = altitude_helper.new_spherical_project_lat_lon(
    full_azimuth_yknf, full_elevation_yknf,
    lat_cam_yknf, lon_cam_yknf,
    H_REF
)

fsmi_lat_ref, fsmi_lon_ref = altitude_helper.new_spherical_project_lat_lon(
    full_azimuth_fsmi, full_elevation_fsmi,
    lat_cam_fsmi, lon_cam_fsmi,
    H_REF
)

GLOBAL_LAT_MIN = int(np.floor(min(np.nanmin(yknf_lat_ref), np.nanmin(fsmi_lat_ref))))
GLOBAL_LAT_MAX = int(np.ceil(max(np.nanmax(yknf_lat_ref), np.nanmax(fsmi_lat_ref))))

global_lat_arr = np.arange(
    GLOBAL_LAT_MIN,
    GLOBAL_LAT_MAX + lat_step,
    lat_step
)

In [ ]:
for h_target in h_target_arr:
    global_lon_arr = [lon_anchor] # MODIFIED SO NOT ALREADY PROJECTING (shifting in the box was caused by projecting TWICE! once here and once in project_lon_slices_and_box (all projection happening in that func now)
    
    #--------project yknf and fsmi to each h_target-----------#
    #----- YKNF ------ #
    yknf_lat_proj_arr, yknf_lon_proj_arr = altitude_helper.new_spherical_project_lat_lon(full_azimuth_yknf, 
                                                                         full_elevation_yknf,
                                                                         lat_cam_yknf,
                                                                         lon_cam_yknf,
                                                                         h_target,
                                                                        )
    
    #----- FSMI ------ #
    fsmi_lat_proj_arr, fsmi_lon_proj_arr = altitude_helper.new_spherical_project_lat_lon(full_azimuth_fsmi, 
                                                                             full_elevation_fsmi,
                                                                             lat_cam_fsmi,
                                                                             lon_cam_fsmi,
                                                                             h_target,
                                                                            )

    
    ### CHANGED: instead of passing in rgb_yknf, rgb_fsmi, pass in the background subtracted version
    yknf_lat_grid, yknf_R_intensity_interp = get_lon_intensity_slice(yknf_lat_proj_arr, yknf_lon_proj_arr, rgb_yknf, 
                                                                 lat_box_max, lat_box_min, lon_box_max, lon_box_min,
                                                                 lat_cam_yknf, lon_cam_yknf,
                                                                 "YKNF", time_index, 
                                                                 H_REF, h_target,
                                                                 global_lon_arr, global_lat_arr)

    #---------intensity slice plot for each lon_to_slice---------#
    fsmi_lat_grid, fsmi_R_intensity_interp = get_lon_intensity_slice(fsmi_lat_proj_arr, fsmi_lon_proj_arr, rgb_fsmi,
                                                                 lat_box_max, lat_box_min, lon_box_max, lon_box_min,
                                                                 lat_cam_fsmi, lon_cam_fsmi, 
                                                                "FSMI", time_index,
                                                                 H_REF, h_target,
                                                                global_lon_arr, global_lat_arr)

    #---------- overlaid yknf and fsmi plot of r channel intensity over latitude at a given projected-up longitude------------#
    plot_lon_intensity_slices(yknf_lat_grid, yknf_R_intensity_interp,
                        fsmi_lat_grid, fsmi_R_intensity_interp,
                        lon_to_slice, h_target)

    #------------------ overlaid plots again of just background subtracted versions of yknf and fsmi plots------------#
    window_size = 9
    yknf_median_arr = scipy.signal.medfilt(yknf_R_intensity_interp, kernel_size=window_size)
    fsmi_median_arr = scipy.signal.medfilt(fsmi_R_intensity_interp, kernel_size=window_size)
    plot_lon_intensity_slices(yknf_lat_grid, yknf_R_intensity_interp-yknf_median_arr,
                        fsmi_lat_grid, fsmi_R_intensity_interp-fsmi_median_arr,
                        lon_to_slice, h_target)

    # -----------individual yknf and fsmi plot comparing original and background subtracted versions of each-------------#
    yknf_median_arr = median_1d_filter_plot(yknf_R_intensity_interp, yknf_lat_grid, "YKNF", 9)
    fsmi_median_arr = median_1d_filter_plot(fsmi_R_intensity_interp, fsmi_lat_grid, "FSMI", 9)


